# Step 1: Clone
increased timeout for retry to fectch the faild repos


In [1]:
from __future__ import annotations

import base64
import csv
import os
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Set
from urllib.parse import urlparse

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT       = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3")
CLONE_ROOT      = Path(r"C:\Clone_retry_repos")

# ORIGINAL MANIFEST (READ ONLY INPUT)
MANIFEST_IN     = WORK_ROOT / "clones_manifest_V1.csv"

# OUTPUT (NEW FILE; will not overwrite input)
MANIFEST_OUT    = WORK_ROOT / f"clones_manifest_retry_{datetime.now():%Y%m%d_%H%M%S}.csv"

# Tokens
TOKENS_ENV_FILE = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
TOKEN_KEYS      = [f"GITHUB_TOKEN_{i}" for i in range(1, 7)]

# Clone options
FETCH_PR_REFS   = True

# Retry behavior
FORCE_RECLONE_ON_RETRY = True   # True = delete repo folder before retrying (recommended for previously-failed partial clones)
RESUME_USING_OUTPUT    = True   # True = if you stop & rerun THE SAME script run, it can skip URLs already written to MANIFEST_OUT

# Timeouts / pacing
DEFAULT_TIMEOUT = 1800          # seconds
SLEEP_BETWEEN_REPOS = 0.0       # seconds; set e.g. 1.0 if you want to be gentle

# -----------------------------
# Helpers
# -----------------------------
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")

def load_env_file(path: Path) -> None:
    """
    Loads KEY=VALUE lines from a .env-like file into os.environ (non-destructive).
    Supports:
      GITHUB_TOKEN_1=...
      export GITHUB_TOKEN_2="..."
      # comments
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path}")
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and v and k not in os.environ:
            os.environ[k] = v

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)  # SSH
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def url_host_and_mode(url: str) -> Tuple[Optional[str], str]:
    u = url.strip()
    if re.match(r"^[^@]+@([^:]+):", u):
        host = u.split("@", 1)[1].split(":", 1)[0]
        return host, "ssh"
    if "://" in u:
        parsed = urlparse(u)
        host = (parsed.netloc or "").split("@")[-1].split(":")[0] or None
        return host, parsed.scheme.lower()
    return None, "other"

def git_auth_config_args(host: str, token: str) -> List[str]:
    basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
    return ["-c", f"http.https://{host}/.extraheader=Authorization: Basic {basic}"]

def sh(
    cmd: List[str],
    cwd: Optional[Path] = None,
    check: bool = True,
    capture: bool = True,
    timeout: Optional[int] = DEFAULT_TIMEOUT,
    auth_url: Optional[str] = None,
    token: Optional[str] = None
) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)

    if cmd and cmd[0] == "git" and token and auth_url:
        host, mode = url_host_and_mode(auth_url)
        if host and mode in ("https", "http"):
            cmd = ["git", *git_auth_config_args(host, token), *cmd[1:]]

    return subprocess.run(
        cmd, cwd=cwd, check=check,
        capture_output=capture, text=True,
        timeout=timeout, env=env
    )

class TokenRotator:
    def __init__(self, tokens: List[str]) -> None:
        self.tokens = tokens[:]
        self.i = 0

    def next(self) -> Tuple[int, str]:
        if not self.tokens:
            raise RuntimeError("No tokens available")
        idx = self.i % len(self.tokens)
        self.i += 1
        return (idx + 1, self.tokens[idx])

def is_retryable_auth_error(msg: str) -> bool:
    m = (msg or "").lower()
    needles = [
        "rate limit", "abuse detection", "too many requests", "http 429",
        "http 403", "403 forbidden", "http 401", "401 unauthorized",
        "authentication failed", "could not read username", "access denied",
        "fatal: unable to access", "remote: permission to"
    ]
    return any(n in m for n in needles)

def is_definitely_not_retryable(msg: str) -> bool:
    m = (msg or "").lower()
    needles = [
        "repository not found",
        "not found",
        "does not exist",
        "could not resolve host",   # sometimes transient, but often DNS/network config
        "invalid username or password",
    ]
    return any(n in m for n in needles)

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

def safe_rmtree(path: Path) -> None:
    # Windows-safe delete: try a couple times
    if not path.exists():
        return
    for _ in range(3):
        try:
            # Make files writable
            for p in path.rglob("*"):
                try:
                    if p.is_file():
                        os.chmod(p, 0o666)
                except Exception:
                    pass
            import shutil
            shutil.rmtree(path, ignore_errors=False)
            return
        except Exception:
            time.sleep(1.0)
    # final attempt ignore errors
    import shutil
    shutil.rmtree(path, ignore_errors=True)

def ensure_full_clone_no_submodules_no_lfs(
    url: str,
    dest_root: Path,
    fetch_pr_refs: bool,
    token: Optional[str] = None
) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if FORCE_RECLONE_ON_RETRY and d.exists():
        safe_rmtree(d)

    if not (d.exists() and (d / ".git").exists()):
        sh(
            ["git", "clone", "--no-single-branch", "--no-recurse-submodules", "--tags", "--quiet", url, str(d)],
            capture=True, auth_url=url, token=token
        )
    else:
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(["git", "config", "core.longpaths", "true"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False, capture=True, auth_url=url, token=token)
    if cp.returncode == 0 and (cp.stdout or "").strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags", "--quiet"], cwd=d, capture=True, auth_url=url, token=token)
    else:
        sh(["git", "fetch", "--tags", "--quiet"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(
        ["git", "fetch", "origin", "--prune", "--tags",
         "+refs/heads/*:refs/remotes/origin/*", "--quiet"],
        cwd=d, check=False, capture=True, auth_url=url, token=token
    )

    if fetch_pr_refs:
        sh(
            ["git", "fetch", "origin",
             "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"],
            cwd=d, check=False, capture=True, auth_url=url, token=token
        )

    return d

# --- Key fix: carry meta even on failure ---
@dataclass
class CloneFailure(Exception):
    url: str
    meta: Dict[str, object]
    err: str

def clone_with_token_rotation(
    url: str,
    clone_root: Path,
    fetch_pr_refs: bool,
    rotator: Optional[TokenRotator],
    max_token_attempts: int = 6
) -> Tuple[Path, Dict[str, object]]:
    host, mode = url_host_and_mode(url)
    meta: Dict[str, object] = {"auth_mode": None, "token_slot": None, "attempts": 0}

    try:
        if mode == "ssh":
            meta["auth_mode"] = "ssh"
            meta["attempts"] = 1
            d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
            return d, meta

        if rotator and mode in ("https", "http") and host:
            meta["auth_mode"] = "https-token"
            tries = min(max_token_attempts, len(rotator.tokens))
            last_err_txt = ""

            for _ in range(tries):
                slot, token = rotator.next()
                meta["attempts"] += 1
                meta["token_slot"] = slot
                try:
                    d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=token)
                    return d, meta
                except subprocess.CalledProcessError as e:
                    err_txt = (e.stderr or e.stdout or str(e)).strip()[:5000]
                    last_err_txt = err_txt

                    if is_retryable_auth_error(err_txt):
                        log(f"Retryable auth/rate error for {url} using token slot {slot}; rotating token...")
                        continue

                    raise  # non retryable -> handled below

            # exhausted tokens
            raise CloneFailure(url=url, meta=meta, err=last_err_txt or "Clone failed after token rotation attempts")

        meta["auth_mode"] = "https-no-token"
        meta["attempts"] = 1
        d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
        return d, meta

    except subprocess.CalledProcessError as e:
        err_txt = (e.stderr or e.stdout or str(e)).strip()[:5000]
        raise CloneFailure(url=url, meta=meta, err=err_txt) from e
    except subprocess.TimeoutExpired as e:
        raise CloneFailure(url=url, meta=meta, err=f"TIMEOUT: {e}") from e
    except CloneFailure:
        raise
    except Exception as e:
        raise CloneFailure(url=url, meta=meta, err=str(e)[:5000]) from e

def load_failed_urls_from_manifest(manifest_path: Path) -> List[str]:
    assert manifest_path.exists(), f"Manifest not found: {manifest_path}"
    urls: List[str] = []
    with manifest_path.open(newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        # Expect at least: repo_url, status
        for row in r:
            url = (row.get("repo_url") or "").strip()
            status = (row.get("status") or "").strip().lower()
            if not url:
                continue
            if status != "ok":
                urls.append(url)
    # de-dupe while keeping order
    seen: Set[str] = set()
    out: List[str] = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def already_done_ok_from_output(out_path: Path) -> Set[str]:
    done: Set[str] = set()
    if not out_path.exists():
        return done
    with out_path.open(newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            url = (row.get("repo_url") or "").strip()
            status = (row.get("status") or "").strip().lower()
            if url and status == "ok":
                done.add(url)
    return done

# -----------------------------
# Main
# -----------------------------
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# Safety: NEVER allow output to overwrite input
if MANIFEST_OUT.resolve() == MANIFEST_IN.resolve():
    raise RuntimeError("MANIFEST_OUT must be different from MANIFEST_IN (refusing to overwrite input).")

# Load tokens
rotator: Optional[TokenRotator] = None
try:
    load_env_file(TOKENS_ENV_FILE)
    tokens = [os.environ.get(k) for k in TOKEN_KEYS]
    tokens = [t for t in tokens if t and t.strip()]
    if tokens:
        rotator = TokenRotator(tokens)
        log(f"Loaded {len(tokens)} GitHub token(s) from {TOKENS_ENV_FILE.name} (slots: 1..{len(tokens)})")
    else:
        log(f"No tokens found in {TOKENS_ENV_FILE.name}; proceeding without tokens.")
except Exception as e:
    log(f"Token file not loaded ({e}); proceeding without tokens.")

# Read failed URLs from ORIGINAL manifest (read-only)
failed_urls = load_failed_urls_from_manifest(MANIFEST_IN)
log(f"Retry mode: FAILED ONLY. Found {len(failed_urls)} repo(s) to retry from: {MANIFEST_IN.name}")

# Optional resume (does NOT touch original manifest)
done_ok = already_done_ok_from_output(MANIFEST_OUT) if RESUME_USING_OUTPUT else set()
if done_ok:
    log(f"Resume: {len(done_ok)} repo(s) already OK in current output file; they will be skipped.")

# Prepare output CSV (append-safe)
fieldnames = [
    "repo_url","dir","status","seconds","total_commits","error",
    "auth_mode","token_slot","attempts"
]

new_file = not MANIFEST_OUT.exists()
with MANIFEST_OUT.open("a", newline="", encoding="utf-8") as f_out:
    w = csv.DictWriter(f_out, fieldnames=fieldnames)
    if new_file:
        w.writeheader()
        f_out.flush()

    ok, fail, skipped = 0, 0, 0

    for url in failed_urls:
        if url in done_ok:
            skipped += 1
            continue

        t0 = time.time()
        rec: Dict[str, object] = {
            "repo_url": url,
            "dir": "",
            "status": "unknown",
            "seconds": "",
            "total_commits": "",
            "error": "",
            "auth_mode": "",
            "token_slot": "",
            "attempts": "",
        }

        try:
            d, meta = clone_with_token_rotation(
                url=url,
                clone_root=CLONE_ROOT,
                fetch_pr_refs=FETCH_PR_REFS,
                rotator=rotator,
                max_token_attempts=6
            )
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            rec["auth_mode"] = meta.get("auth_mode")
            rec["token_slot"] = meta.get("token_slot")
            rec["attempts"] = meta.get("attempts")
            ok += 1

            log(f"[ok] {url} -> {rec['dir']}  commits={rec['total_commits']}  "
                f"auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")

        except CloneFailure as e:
            rec["status"] = "error"
            rec["error"] = (e.err or "")[:2000]
            rec["auth_mode"] = e.meta.get("auth_mode")
            rec["token_slot"] = e.meta.get("token_slot")
            rec["attempts"] = e.meta.get("attempts")
            fail += 1

            # Helpful print
            log(f"[error] {url}  auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")
            # If you want to see a short reason:
            short = (rec["error"] or "").replace("\n", " ")[:180]
            if short:
                log(f"        reason: {short}")

        rec["seconds"] = round(time.time() - t0, 2)

        # Write one row immediately so you don't lose progress if you stop the script
        w.writerow(rec)
        f_out.flush()

        if SLEEP_BETWEEN_REPOS > 0:
            time.sleep(SLEEP_BETWEEN_REPOS)

log(f"Done. OK={ok}, FAIL={fail}, SKIPPED={skipped}. Output manifest: {MANIFEST_OUT}")


[2025-12-15 22:23:31] Loaded 6 GitHub token(s) from All_Tokens.env (slots: 1..6)
[2025-12-15 22:23:31] Retry mode: FAILED ONLY. Found 21 repo(s) to retry from: clones_manifest_V1.csv
[2025-12-15 22:23:33] [ok] https://github.com/jakenjarvis/Android-OrmLiteContentProvider -> C:\Clone_retry_repos\jakenjarvis__Android-OrmLiteContentProvider  commits=250  auth=https-token token_slot=1 attempts=1
[2025-12-15 22:23:35] [ok] https://github.com/ksoichiro/AndroidFormEnhancer -> C:\Clone_retry_repos\ksoichiro__AndroidFormEnhancer  commits=149  auth=https-token token_slot=2 attempts=1
[2025-12-15 22:23:41] [ok] https://github.com/ksoichiro/Android-ObservableScrollView -> C:\Clone_retry_repos\ksoichiro__Android-ObservableScrollView  commits=430  auth=https-token token_slot=3 attempts=1
[2025-12-15 22:24:49] [ok] https://github.com/sqldelight/sqldelight -> C:\Clone_retry_repos\sqldelight__sqldelight  commits=10374  auth=https-token token_slot=4 attempts=1
[2025-12-15 22:43:21] [ok] https://github.c

In [ ]:
#Clone single

In [2]:
from __future__ import annotations

import base64
import csv
import os
import re
import subprocess
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Set
from urllib.parse import urlparse

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT       = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3")

# Clone just this one repo
SINGLE_URL      = "https://github.com/zserge/log"

# OUTPUT (NEW FILE)
MANIFEST_OUT    = WORK_ROOT / f"clone_single_{datetime.now():%Y%m%d_%H%M%S}.csv"

# IMPORTANT: clone folder lives in same folder as output CSV (WORK_ROOT)
CLONE_ROOT      = WORK_ROOT / "cloned_single"

# Tokens
TOKENS_ENV_FILE = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
TOKEN_KEYS      = [f"GITHUB_TOKEN_{i}" for i in range(1, 7)]

# Clone options
FETCH_PR_REFS   = True

# Behavior
FORCE_RECLONE_ON_RETRY = True
DEFAULT_TIMEOUT = 1800
SLEEP_BETWEEN_REPOS = 0.0

# -----------------------------
# Helpers
# -----------------------------
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def log(msg: str) -> None:
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}")

def load_env_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path}")
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.lower().startswith("export "):
            line = line[7:].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and v and k not in os.environ:
            os.environ[k] = v

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)  # SSH
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def url_host_and_mode(url: str) -> Tuple[Optional[str], str]:
    u = url.strip()
    if re.match(r"^[^@]+@([^:]+):", u):
        host = u.split("@", 1)[1].split(":", 1)[0]
        return host, "ssh"
    if "://" in u:
        parsed = urlparse(u)
        host = (parsed.netloc or "").split("@")[-1].split(":")[0] or None
        return host, parsed.scheme.lower()
    return None, "other"

def git_auth_config_args(host: str, token: str) -> List[str]:
    basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
    return ["-c", f"http.https://{host}/.extraheader=Authorization: Basic {basic}"]

def sh(
    cmd: List[str],
    cwd: Optional[Path] = None,
    check: bool = True,
    capture: bool = True,
    timeout: Optional[int] = DEFAULT_TIMEOUT,
    auth_url: Optional[str] = None,
    token: Optional[str] = None
) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)

    if cmd and cmd[0] == "git" and token and auth_url:
        host, mode = url_host_and_mode(auth_url)
        if host and mode in ("https", "http"):
            cmd = ["git", *git_auth_config_args(host, token), *cmd[1:]]

    return subprocess.run(
        cmd, cwd=cwd, check=check,
        capture_output=capture, text=True,
        timeout=timeout, env=env
    )

class TokenRotator:
    def __init__(self, tokens: List[str]) -> None:
        self.tokens = tokens[:]
        self.i = 0

    def next(self) -> Tuple[int, str]:
        if not self.tokens:
            raise RuntimeError("No tokens available")
        idx = self.i % len(self.tokens)
        self.i += 1
        return (idx + 1, self.tokens[idx])

def is_retryable_auth_error(msg: str) -> bool:
    m = (msg or "").lower()
    needles = [
        "rate limit", "abuse detection", "too many requests", "http 429",
        "http 403", "403 forbidden", "http 401", "401 unauthorized",
        "authentication failed", "could not read username", "access denied",
        "fatal: unable to access", "remote: permission to"
    ]
    return any(n in m for n in needles)

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

def safe_rmtree(path: Path) -> None:
    if not path.exists():
        return
    for _ in range(3):
        try:
            for p in path.rglob("*"):
                try:
                    if p.is_file():
                        os.chmod(p, 0o666)
                except Exception:
                    pass
            import shutil
            shutil.rmtree(path, ignore_errors=False)
            return
        except Exception:
            time.sleep(1.0)
    import shutil
    shutil.rmtree(path, ignore_errors=True)

def ensure_full_clone_no_submodules_no_lfs(
    url: str,
    dest_root: Path,
    fetch_pr_refs: bool,
    token: Optional[str] = None
) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if FORCE_RECLONE_ON_RETRY and d.exists():
        safe_rmtree(d)

    if not (d.exists() and (d / ".git").exists()):
        sh(
            ["git", "clone", "--no-single-branch", "--no-recurse-submodules", "--tags", "--quiet", url, str(d)],
            capture=True, auth_url=url, token=token
        )
    else:
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(["git", "config", "core.longpaths", "true"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False, capture=True, auth_url=url, token=token)
    if cp.returncode == 0 and (cp.stdout or "").strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags", "--quiet"], cwd=d, capture=True, auth_url=url, token=token)
    else:
        sh(["git", "fetch", "--tags", "--quiet"], cwd=d, check=False, capture=True, auth_url=url, token=token)

    sh(
        ["git", "fetch", "origin", "--prune", "--tags",
         "+refs/heads/*:refs/remotes/origin/*", "--quiet"],
        cwd=d, check=False, capture=True, auth_url=url, token=token
    )

    if fetch_pr_refs:
        sh(
            ["git", "fetch", "origin",
             "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"],
            cwd=d, check=False, capture=True, auth_url=url, token=token
        )

    return d

@dataclass
class CloneFailure(Exception):
    url: str
    meta: Dict[str, object]
    err: str

def clone_with_token_rotation(
    url: str,
    clone_root: Path,
    fetch_pr_refs: bool,
    rotator: Optional[TokenRotator],
    max_token_attempts: int = 6
) -> Tuple[Path, Dict[str, object]]:
    host, mode = url_host_and_mode(url)
    meta: Dict[str, object] = {"auth_mode": None, "token_slot": None, "attempts": 0}

    try:
        if mode == "ssh":
            meta["auth_mode"] = "ssh"
            meta["attempts"] = 1
            d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
            return d, meta

        if rotator and mode in ("https", "http") and host:
            meta["auth_mode"] = "https-token"
            tries = min(max_token_attempts, len(rotator.tokens))
            last_err_txt = ""

            for _ in range(tries):
                slot, token = rotator.next()
                meta["attempts"] += 1
                meta["token_slot"] = slot
                try:
                    d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=token)
                    return d, meta
                except subprocess.CalledProcessError as e:
                    err_txt = (e.stderr or e.stdout or str(e)).strip()[:5000]
                    last_err_txt = err_txt
                    if is_retryable_auth_error(err_txt):
                        log(f"Retryable auth/rate error for {url} using token slot {slot}; rotating token...")
                        continue
                    raise

            raise CloneFailure(url=url, meta=meta, err=last_err_txt or "Clone failed after token rotation attempts")

        meta["auth_mode"] = "https-no-token"
        meta["attempts"] = 1
        d = ensure_full_clone_no_submodules_no_lfs(url, clone_root, fetch_pr_refs, token=None)
        return d, meta

    except subprocess.CalledProcessError as e:
        err_txt = (e.stderr or e.stdout or str(e)).strip()[:5000]
        raise CloneFailure(url=url, meta=meta, err=err_txt) from e
    except subprocess.TimeoutExpired as e:
        raise CloneFailure(url=url, meta=meta, err=f"TIMEOUT: {e}") from e
    except CloneFailure:
        raise
    except Exception as e:
        raise CloneFailure(url=url, meta=meta, err=str(e)[:5000]) from e

# -----------------------------
# Main
# -----------------------------
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# Load tokens
rotator: Optional[TokenRotator] = None
try:
    load_env_file(TOKENS_ENV_FILE)
    tokens = [os.environ.get(k) for k in TOKEN_KEYS]
    tokens = [t for t in tokens if t and t.strip()]
    if tokens:
        rotator = TokenRotator(tokens)
        log(f"Loaded {len(tokens)} GitHub token(s) from {TOKENS_ENV_FILE.name} (slots: 1..{len(tokens)})")
    else:
        log(f"No tokens found in {TOKENS_ENV_FILE.name}; proceeding without tokens.")
except Exception as e:
    log(f"Token file not loaded ({e}); proceeding without tokens.")

fieldnames = [
    "repo_url","dir","status","seconds","total_commits","error",
    "auth_mode","token_slot","attempts"
]

with MANIFEST_OUT.open("w", newline="", encoding="utf-8") as f_out:
    w = csv.DictWriter(f_out, fieldnames=fieldnames)
    w.writeheader()
    f_out.flush()

    url = SINGLE_URL
    t0 = time.time()

    rec: Dict[str, object] = {
        "repo_url": url,
        "dir": "",
        "status": "unknown",
        "seconds": "",
        "total_commits": "",
        "error": "",
        "auth_mode": "",
        "token_slot": "",
        "attempts": "",
    }

    try:
        d, meta = clone_with_token_rotation(
            url=url,
            clone_root=CLONE_ROOT,
            fetch_pr_refs=FETCH_PR_REFS,
            rotator=rotator,
            max_token_attempts=6
        )
        rec["dir"] = str(d)
        rec["total_commits"] = get_total_commits(d)
        rec["status"] = "ok"
        rec["auth_mode"] = meta.get("auth_mode")
        rec["token_slot"] = meta.get("token_slot")
        rec["attempts"] = meta.get("attempts")

        log(f"[ok] {url} -> {rec['dir']}  commits={rec['total_commits']}  "
            f"auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")

    except CloneFailure as e:
        rec["status"] = "error"
        rec["error"] = (e.err or "")[:2000]
        rec["auth_mode"] = e.meta.get("auth_mode")
        rec["token_slot"] = e.meta.get("token_slot")
        rec["attempts"] = e.meta.get("attempts")

        log(f"[error] {url}  auth={rec['auth_mode']} token_slot={rec['token_slot']} attempts={rec['attempts']}")
        short = (rec["error"] or "").replace("\n", " ")[:180]
        if short:
            log(f"        reason: {short}")

    rec["seconds"] = round(time.time() - t0, 2)

    w.writerow(rec)
    f_out.flush()

log(f"Done. Output manifest: {MANIFEST_OUT}")
log(f"Clones saved under: {CLONE_ROOT}")


[2025-12-17 21:55:36] Loaded 6 GitHub token(s) from All_Tokens.env (slots: 1..6)
[2025-12-17 21:55:38] [ok] https://github.com/zserge/log -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\cloned_single\zserge__log  commits=44  auth=https-token token_slot=1 attempts=1
[2025-12-17 21:55:38] Done. Output manifest: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\clone_single_20251217_215536.csv
[2025-12-17 21:55:38] Clones saved under: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\cloned_single
